In [ ]:
!pip install pyspark

In [ ]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName('Transformation').getOrCreate()
print("spark")

spark


In [ ]:
from google.colab import files

uploaded = files.upload()

Saving lazy_execution_sales_25000.csv to lazy_execution_sales_25000.csv


In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col,
    sum,
    count,
    round,
    desc,
    asc
)

spark = SparkSession.builder \
    .appName("TransformationsAndActions") \
    .getOrCreate()

# Load sales data
sales_df = spark.read.csv(
    "lazy_execution_sales_25000.csv",
    header=True,
    inferSchema=True
)

# Load products data
products_df = spark.read.csv(
    "lazy_execution_products.csv",
    header=True,
    inferSchema=True
)

# Load stores data
stores_df = spark.read.csv(
    "lazy_execution_stores.csv",
    header=True,
    inferSchema=True
)

# Display schemas
sales_df.printSchema()
products_df.printSchema()
stores_df.printSchema()

# Check the number of records
print("Sales records:", sales_df.count())
print("Product records:", products_df.count())
print("Store records:", stores_df.count())

root
 |-- order_id: string (nullable = true)
 |-- order_date: date (nullable = true)
 |-- store_id: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- unit_price: integer (nullable = true)
 |-- discount_pct: integer (nullable = true)
 |-- payment_method: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- sales_channel: string (nullable = true)

root
 |-- product_id: string (nullable = true)
 |-- product_name: string (nullable = true)
 |-- category: string (nullable = true)
 |-- list_price: integer (nullable = true)
 |-- cost_price: integer (nullable = true)

root
 |-- store_id: string (nullable = true)
 |-- city: string (nullable = true)
 |-- region: string (nullable = true)
 |-- store_type: string (nullable = true)

Sales records: 25000
Product records: 15
Store records: 8


In [ ]:
#1
filtered_df = sales_df.filter(col("quantity") > 5)


#2
selected_df = filtered_df.select(
    "order_id",
    "product_id",
    "quantity",
    "unit_price"
)

selected_df.show()


#3
expensive_df = sales_df.filter(
    col("unit_price") > 20000
)

total_records = expensive_df.count()
print(total_records)

#4
gross_df = sales_df.withColumn(
    "gross_amount",
    col("quantity") * col("unit_price")
)

# Inspect the execution plan
gross_df.explain()

# Display results
gross_df.show()


# Exercise 5: Filter completed orders and sort by price

completed_sorted_df = sales_df.filter(
    col("order_status") == "Completed"
).orderBy(
    col("unit_price").desc()
)

# Retrieve the first 10 records
records = completed_sorted_df.take(10)

for record in records:
    print(record)




+--------+----------+--------+----------+
|order_id|product_id|quantity|unit_price|
+--------+----------+--------+----------+
| O000002|      P014|       6|     29999|
| O000003|      P008|       6|     22999|
| O000004|      P008|       7|     22999|
| O000006|      P005|       8|      2499|
| O000007|      P010|       8|      3999|
| O000008|      P002|       9|      2999|
| O000012|      P010|       9|      3999|
| O000013|      P013|       6|      5999|
| O000014|      P005|       8|      2499|
| O000015|      P007|       7|     79999|
| O000017|      P004|       9|     16999|
| O000018|      P012|       7|      6999|
| O000020|      P010|       7|      3999|
| O000021|      P006|       7|     55999|
| O000022|      P003|       7|      1899|
| O000023|      P010|       9|      3999|
| O000024|      P009|       7|      1999|
| O000028|      P014|       7|     29999|
| O000033|      P002|       6|      2999|
| O000034|      P012|       6|      6999|
+--------+----------+--------+----

part b

In [ ]:
# Exercise 6: Chain transformations with one final action

pipeline_df = sales_df.filter(
    col("order_status") == "Completed"
).withColumn(
    "gross_amount",
    col("quantity") * col("unit_price")
).select(
    "order_id",
    "product_id",
    "quantity",
    "unit_price",
    "gross_amount"
).orderBy(
    col("gross_amount").desc()
)

# Single action at the end
pipeline_df.show()

+--------+----------+--------+----------+------------+
|order_id|product_id|quantity|unit_price|gross_amount|
+--------+----------+--------+----------+------------+
| O002603|      P007|       9|     79999|      719991|
| O006170|      P007|       9|     79999|      719991|
| O002693|      P007|       9|     79999|      719991|
| O000107|      P007|       9|     79999|      719991|
| O002841|      P007|       9|     79999|      719991|
| O000367|      P007|       9|     79999|      719991|
| O002940|      P007|       9|     79999|      719991|
| O000527|      P007|       9|     79999|      719991|
| O003108|      P007|       9|     79999|      719991|
| O000690|      P007|       9|     79999|      719991|
| O003542|      P007|       9|     79999|      719991|
| O001201|      P007|       9|     79999|      719991|
| O003884|      P007|       9|     79999|      719991|
| O001300|      P007|       9|     79999|      719991|
| O003897|      P007|       9|     79999|      719991|
| O001796|

In [ ]:
# Exercise 7: Calculate discounted sales amount

discounted_df = sales_df.filter(
    col("discount_pct") >= 10
).withColumn(
    "discounted_sales_amount",
    col("quantity") *
    col("unit_price") *
    (1 - col("discount_pct") / 100)
).orderBy(
    col("discounted_sales_amount").desc()
)

discounted_df.show(15)

+--------+----------+--------+----------+--------+----------+------------+--------------+------------+-------------+-----------------------+
|order_id|order_date|store_id|product_id|quantity|unit_price|discount_pct|payment_method|order_status|sales_channel|discounted_sales_amount|
+--------+----------+--------+----------+--------+----------+------------+--------------+------------+-------------+-----------------------+
| O006653|2026-06-29|     S01|      P007|       9|     79999|          10|          Card|   Cancelled|       Online|               647991.9|
| O011893|2026-03-26|     S07|      P007|       9|     79999|          10|           UPI|   Completed|       Online|               647991.9|
| O006699|2026-06-25|     S05|      P007|       9|     79999|          10|   Net Banking|   Completed|       Online|               647991.9|
| O003108|2026-02-23|     S08|      P007|       9|     79999|          10|   Net Banking|   Completed|     In Store|               647991.9|
| O007743|202

In [ ]:
# Exercise 8: Total quantity sold by payment method

payment_summary_df = sales_df.groupBy(
    "payment_method"
).agg(
    sum("quantity").alias("total_quantity_sold")
).orderBy(
    col("total_quantity_sold").desc()
)

payment_summary_df.show()

+--------------+-------------------+
|payment_method|total_quantity_sold|
+--------------+-------------------+
|          Cash|              31754|
|          Card|              31327|
|           UPI|              30894|
|   Net Banking|              30889|
+--------------+-------------------+



In [ ]:
# Exercise 9: Count transactions by channel and status

channel_status_df = sales_df.groupBy(
    "sales_channel",
    "order_status"
).agg(
    count("*").alias("transaction_count")
).orderBy(
    "sales_channel",
    "order_status"
)

channel_status_df.show()

+-------------+------------+-----------------+
|sales_channel|order_status|transaction_count|
+-------------+------------+-----------------+
|     In Store|   Cancelled|              592|
|     In Store|   Completed|            10978|
|     In Store|    Returned|             1013|
|       Online|   Cancelled|              633|
|       Online|   Completed|            10797|
|       Online|    Returned|              987|
+-------------+------------+-----------------+



In [ ]:
# Exercise 10: Select columns, drop store_id, and retrieve one row

orders_df = sales_df.select(
    "order_id",
    "store_id",
    "product_id",
    "quantity"
)

orders_without_store_df = orders_df.drop("store_id")

first_record = orders_without_store_df.first()

print("First record:")
print(first_record)

First record:
Row(order_id='O000001', product_id='P002', quantity=3)


part c

In [ ]:
# Exercise 11: Join sales and products

sales_products_df = sales_df.join(
    products_df,
    on="product_id",
    how="inner"
).select(
    "product_name",
    "category",
    "quantity",
    "unit_price"
)

sales_products_df.show(20)

+-------------------+-----------+--------+----------+
|       product_name|   category|quantity|unit_price|
+-------------------+-----------+--------+----------+
|Mechanical Keyboard|Accessories|       3|      2999|
|      Smartphone 5G|    Mobiles|       6|     29999|
|     Android Tablet|    Tablets|       6|     22999|
|     Android Tablet|    Tablets|       7|     22999|
|          Laptop 16|  Computers|       2|     79999|
|          Webcam HD|Accessories|       8|      2499|
|  Bluetooth Speaker|      Audio|       8|      3999|
|Mechanical Keyboard|Accessories|       9|      2999|
|          Laptop 16|  Computers|       5|     79999|
|          Laptop 14|  Computers|       5|     55999|
|          Laptop 16|  Computers|       1|     79999|
|  Bluetooth Speaker|      Audio|       9|      3999|
|   External HDD 2TB|    Storage|       6|      5999|
|          Webcam HD|Accessories|       8|      2499|
|          Laptop 16|  Computers|       7|     79999|
|          Laptop 14|  Compu

In [ ]:
# Exercise 12: Total quantity sold by city

city_sales_df = sales_df.join(
    stores_df,
    on="store_id",
    how="inner"
).groupBy(
    "city"
).agg(
    sum("quantity").alias("total_quantity_sold")
).orderBy(
    col("total_quantity_sold").desc()
)

city_sales_df.show()

+---------+-------------------+
|     city|total_quantity_sold|
+---------+-------------------+
|   Jaipur|              16019|
|Hyderabad|              15784|
|  Chennai|              15727|
|Bengaluru|              15718|
|  Kolkata|              15619|
|   Mumbai|              15489|
|     Pune|              15407|
|    Delhi|              15101|
+---------+-------------------+



In [ ]:
# Exercise 13: Completed-order revenue by category and region

all_data_df = sales_df.join(
    products_df,
    on="product_id",
    how="inner"
).join(
    stores_df,
    on="store_id",
    how="inner"
)

category_region_revenue_df = all_data_df.filter(
    col("order_status") == "Completed"
).withColumn(
    "revenue",
    col("quantity") * col("unit_price")
).groupBy(
    "category",
    "region"
).agg(
    sum("revenue").alias("total_revenue")
).orderBy(
    col("total_revenue").desc()
)

category_region_revenue_df.show()

+-----------+------+-------------+
|   category|region|total_revenue|
+-----------+------+-------------+
|  Computers| South|    371954527|
|  Computers| North|    240756425|
|  Computers|  West|    221092729|
|  Computers|  East|    122990192|
|    Mobiles| South|    120674524|
|    Mobiles|  West|     80372388|
|    Mobiles| North|     75482543|
|    Tablets| South|     64006217|
|   Displays| South|     45761308|
|    Mobiles|  East|     41178126|
|    Tablets| North|     38270336|
|    Tablets|  West|     37488370|
|    Storage| South|     36976319|
|      Audio| South|     34629665|
|   Displays| North|     32468090|
|   Displays|  West|     28286336|
|Accessories| South|     28139018|
|      Audio|  West|     25317111|
|      Audio| North|     24327211|
|    Storage| North|     24150285|
+-----------+------+-------------+
only showing top 20 rows


In [ ]:
# Exercise 14: Top five products by discounted revenue

completed_sales_df = sales_df.filter(
    col("order_status") == "Completed"
)

product_revenue_df = completed_sales_df.join(
    products_df,
    on="product_id",
    how="inner"
).withColumn(
    "discounted_revenue",
    col("quantity") *
    col("unit_price") *
    (1 - col("discount_pct") / 100)
).groupBy(
    "product_id",
    "product_name"
).agg(
    sum("discounted_revenue").alias("total_revenue")
).orderBy(
    col("total_revenue").desc()
)

product_revenue_df.show(5)

+----------+---------------+--------------------+
|product_id|   product_name|       total_revenue|
+----------+---------------+--------------------+
|      P007|      Laptop 16|5.2272546585000014E8|
|      P006|      Laptop 14| 3.830051604999999E8|
|      P014|  Smartphone 5G| 2.060406317500002E8|
|      P008| Android Tablet|1.5040656029999995E8|
|      P004|27-inch Monitor|1.1406074015000013E8|
+----------+---------------+--------------------+
only showing top 5 rows


part d

In [ ]:
# Exercise 15: Observe lazy execution

# Transformation 1: Filter completed orders
step1_df = sales_df.filter(
    col("order_status") == "Completed"
)

print("Transformation 1: Filter completed orders created.")

# Transformation 2: Select required columns
step2_df = step1_df.select(
    "product_id",
    "quantity",
    "unit_price"
)

print("Transformation 2: Selected columns.")

# Transformation 3: Calculate gross amount
step3_df = step2_df.withColumn(
    "gross_amount",
    col("quantity") * col("unit_price")
)

print("Transformation 3: Gross amount column created.")

# Transformation 4: Group by product and calculate totals
step4_df = step3_df.groupBy(
    "product_id"
).agg(
    sum("quantity").alias("total_quantity"),
    sum("gross_amount").alias("total_gross_amount")
)

print("Transformation 4: Grouped by product.")

# Transformation 5: Sort by total gross amount
step5_df = step4_df.orderBy(
    col("total_gross_amount").desc()
)

print("Transformation 5: Sorting transformation created.")

# Trigger execution with an action
step5_df.show()

Transformation 1: Filter completed orders created.
Transformation 2: Selected columns.
Transformation 3: Gross amount column created.
Transformation 4: Grouped by product.
Transformation 5: Sorting transformation created.
+----------+--------------+------------------+
|product_id|total_quantity|total_gross_amount|
+----------+--------------+------------------+
|      P007|          6904|         552313096|
|      P006|          7223|         404480777|
|      P014|          7241|         217222759|
|      P008|          6894|         158555106|
|      P004|          7067|         120131933|
|      P015|          7178|         100484822|
|      P011|          7364|          66268636|
|      P012|          7441|          52079559|
|      P013|          7478|          44860522|
|      P010|          7514|          30048486|
|      P002|          7109|          21319891|
|      P005|          7518|          18787482|
|      P009|          7155|          14302845|
|      P003|          7157

In [ ]:
# Exercise 16: Explain, show, and count

filtered_completed_df = sales_df.filter(
    col("order_status") == "Completed"
)

# Inspect the execution plan
filtered_completed_df.explain()

print("Execution plan displayed.")

# Action 1: Display records
filtered_completed_df.show()

# Action 2: Count records
completed_count = filtered_completed_df.count()

print("Total completed orders:", completed_count)

== Physical Plan ==
*(1) Filter (isnotnull(order_status#25) AND (order_status#25 = Completed))
+- FileScan csv [order_id#17,order_date#18,store_id#19,product_id#20,quantity#21,unit_price#22,discount_pct#23,payment_method#24,order_status#25,sales_channel#26] Batched: false, DataFilters: [isnotnull(order_status#25), (order_status#25 = Completed)], Format: CSV, Location: InMemoryFileIndex(1 paths)[file:/content/lazy_execution_sales_25000.csv], PartitionFilters: [], PushedFilters: [IsNotNull(order_status), EqualTo(order_status,Completed)], ReadSchema: struct<order_id:string,order_date:date,store_id:string,product_id:string,quantity:int,unit_price:...


Execution plan displayed.
+--------+----------+--------+----------+--------+----------+------------+--------------+------------+-------------+
|order_id|order_date|store_id|product_id|quantity|unit_price|discount_pct|payment_method|order_status|sales_channel|
+--------+----------+--------+----------+--------+----------+------------+---------

In [ ]:
# Exercise 17: Collect grouped quantity by category

category_quantity_df = sales_df.join(
    products_df,
    on="product_id",
    how="inner"
).groupBy(
    "category"
).agg(
    sum("quantity").alias("total_quantity")
)

# Collect the aggregated results
category_results = category_quantity_df.collect()

for row in category_results:
    print(row)

Row(category='Computers', total_quantity=16258)
Row(category='Displays', total_quantity=7870)
Row(category='Mobiles', total_quantity=16572)
Row(category='Accessories', total_quantity=41839)
Row(category='Storage', total_quantity=17251)
Row(category='Audio', total_quantity=17159)
Row(category='Tablets', total_quantity=7915)


In [ ]:
# Exercise 18: Calculate discounted revenue and save as Parquet

discounted_revenue_df = sales_df.filter(
    col("order_status") == "Completed"
).withColumn(
    "discounted_revenue",
    col("quantity") *
    col("unit_price") *
    (1 - col("discount_pct") / 100)
)

print("Filtered completed orders and calculated discounted revenue.")

# Save the DataFrame as Parquet
discounted_revenue_df.write.mode("overwrite").parquet(
    "/content/completed_discounted_revenue"
)

print("Data saved successfully as Parquet.")

Filtered completed orders and calculated discounted revenue.
Data saved successfully as Parquet.


In [ ]:
#19 Read the saved Parquet file
parquet_df = spark.read.parquet(
    "/content/completed_discounted_revenue"
)

parquet_df.show(10)

print("Total saved records:", parquet_df.count())

+--------+----------+--------+----------+--------+----------+------------+--------------+------------+-------------+------------------+
|order_id|order_date|store_id|product_id|quantity|unit_price|discount_pct|payment_method|order_status|sales_channel|discounted_revenue|
+--------+----------+--------+----------+--------+----------+------------+--------------+------------+-------------+------------------+
| O000001|2026-06-22|     S04|      P002|       3|      2999|          10|   Net Banking|   Completed|     In Store|            8097.3|
| O000002|2026-05-29|     S07|      P014|       6|     29999|           0|          Card|   Completed|       Online|          179994.0|
| O000003|2026-06-27|     S06|      P008|       6|     22999|           0|          Card|   Completed|     In Store|          137994.0|
| O000004|2026-04-10|     S03|      P008|       7|     22999|          15|          Cash|   Completed|       Online|         136844.05|
| O000005|2026-03-14|     S06|      P007|       